# Load CSV dataset.

In [ ]:
import glob
import pandas as pd
import numpy as np

# Get iterables for csv datasets.
table_csv_wildcard = "./csv/GuanYinSurfaces/GuanYinTable/*"
wall_csv_wildcard = "./csv/GuanYinSurfaces/GuanYinWall/*"
table_csv_paths = glob.glob(table_csv_wildcard)
wall_csv_paths = glob.glob(wall_csv_wildcard)

# Load the CSV files
table_dataframes = list(map(lambda file_path: pd.read_csv(file_path), table_csv_paths))
wall_dataframes = list(map(lambda file_path: pd.read_csv(file_path), wall_csv_paths))
all_dataframes = table_dataframes + wall_dataframes

for df in all_dataframes:
    # Enforce numbers are numeric.
    assert all(df.dtypes.apply(pd.api.types.is_numeric_dtype))
    # Assert that the data has 7 columns.
    assert df.shape[1] == 7, "Data should have 7 columns."
    # Rename columns.
    df.columns = ["t", "ax", "ay", "az", "gx", "gy", "gz"]
    # Calculate acceleration magnitude.
    df["a_mag"] = (df["ax"] ** 2 + df["ay"] ** 2 + df["az"] ** 2) ** 0.5
    # Calculate sample rate.
    df.attrs["sample_rate"] = 1 / df["t"].diff().median()
    # Calculate fft of entire a_mag.
    df.attrs["a_mag_fft"] = np.fft.fft(df["a_mag"])

for idx, df in enumerate(table_dataframes):
    df.attrs["surface"] = "table " + str(idx)
for idx, df in enumerate(wall_dataframes):
    df.attrs["surface"] = "wall " + str(idx)

# Plot Dataset

In [ ]:
import plotly.graph_objects as go

for idx, df in enumerate(all_dataframes):
    fig = go.Figure()
    fig.add_trace(
        go.Scatter(
            x=df["t"],
            y=df["a_mag"],
            mode="lines",
            line=dict(color="blue"),
        )
    )
    fig.update_layout(
        title="Surface: " + df.attrs["surface"],
        xaxis_title="Time",
        yaxis_title="Accel_Mag",
        height=300,
    )
    fig.show()

    a_mag_fft_mag = np.abs(df.attrs["a_mag_fft"])
    a_mag_fft_frequencies = np.fft.fftfreq(
        len(a_mag_fft_mag), 1 / df.attrs["sample_rate"]
    )
    mask = a_mag_fft_frequencies > 0
    fig = go.Figure()
    fig.add_trace(
        go.Scatter(
            x=a_mag_fft_frequencies[mask],
            y=a_mag_fft_mag[mask],
            mode="lines",
            line=dict(color="orange"),
        )
    )
    fig.update_layout(
        title="Surface: " + df.attrs["surface"],
        xaxis_title="Freq",
        yaxis_title="Accel_Mag",
        height=300,
    )

    fig.show()

# Play sounds at 48kHz

In [ ]:
import sounddevice as sd
from IPython.display import clear_output
import random

# Play the sound
for idx, df in enumerate(random.sample(all_dataframes, len(all_dataframes))):
    # Upsample a_mag.
    target_sample_rate = 48000
    original_sample_rate = float(df.attrs["sample_rate"])

    a_mag = df["a_mag"].to_numpy()
    a_mag_start_time = df["t"][0]
    a_mag_end_time = df["t"].iloc[-1]
    a_mag_new_num_samples = int(len(a_mag) * target_sample_rate / original_sample_rate)
    x_old = np.linspace(a_mag_start_time, a_mag_end_time, len(a_mag))
    x_new = np.linspace(
        a_mag_start_time,
        a_mag_end_time,
        a_mag_new_num_samples,
    )
    a_mag_upsampled = np.interp(x_new, x_old, a_mag)
    a_mag_upsampled_normalized = a_mag_upsampled / max(abs(a_mag_upsampled))

    # Plot waveform.
    fig = go.Figure()
    fig.add_trace(
        go.Scatter(
            x=x_new,
            y=a_mag_upsampled_normalized,
            mode="lines",
            line=dict(color="blue"),
        )
    )
    fig.update_layout(
        title="Surface: " + df.attrs["surface"],
        xaxis_title="time: (48k samples per sec)",
        yaxis_title="a_mag_upsampled_normalized",
    )
    fig.show()

    # Play imu waveform.
    sd.play(a_mag_upsampled_normalized, samplerate=target_sample_rate)
    sd.wait()

    # Clear jupyter output.
    clear_output()